In [ ]:
import pickle

import torch
from torchmetrics.functional import pairwise_cosine_similarity
from transformers import AutoTokenizer
from transformers import T5EncoderModel


In [ ]:


def get_tac_encodings(
        encoder, input_ids: torch.LongTensor, attention_mask: torch.LongTensor, tactic_lens
) -> torch.FloatTensor:
    # encode all tokens with tactic included
    combined_enc = encoder(input_ids, attention_mask, return_dict=True).last_hidden_state

    # get the tactic embeddings and mean pool them using the provided lengths
    tac_enc = []

    for i in range(combined_enc.shape[0]):
        enc = combined_enc[i, :tactic_lens[i]]
        enc = enc.sum(dim=0) / tactic_lens[i]
        enc = F.normalize(enc, dim=0)
        tac_enc.append(enc)

    tac_enc = torch.stack(tac_enc, dim=0).unsqueeze(1)
    return tac_enc


def get_vecs(encoder, tactics, goal, theorem):
    state = goal.data['augmented_state'] if hasattr(goal, 'data') and 'augmented_state' in goal.data else goal.goal

    encs = []

    # todo chunk into batches enc speedup
    for t in tactics:
        goal = [t + theorem + '\n\n' + state]

        tokenized_goals = tokenizer(
            goal,
            padding="longest",
            max_length=int(3000),
            truncation=True,
            return_tensors="pt", )

        tokenized_tactics = tokenizer(
            tactics,
            padding="longest",
            max_length=3000,
            truncation=True,
            return_tensors="pt",
        )

        lens = tokenized_tactics.attention_mask.sum(dim=1)

        enc = get_tac_encodings(encoder, tokenized_goals.input_ids.cuda(), tokenized_goals.attention_mask.cuda(), lens.cuda())

        # scale enc by normalised tactic logprob
        # enc = enc * probs[tactics.index(t)]
        encs.append(enc.squeeze(0).squeeze(0))

    return torch.stack(encs, dim=0)


In [ ]:

import torch.nn.functional as F


def _encode(
        encoder, input_ids: torch.LongTensor, attention_mask: torch.LongTensor
) -> torch.FloatTensor:
    hidden_states = encoder(
        input_ids=input_ids,
        attention_mask=attention_mask,
        return_dict=True,
    ).last_hidden_state

    # Masked average.
    lens = attention_mask.sum(dim=1)
    features = (hidden_states * attention_mask.unsqueeze(2)).sum(
        dim=1
    ) / lens.unsqueeze(1)

    # Normalize the feature vector to have unit norm.
    return F.normalize(features, dim=1)


In [ ]:

def plot(start, end, sims):
    # Convert the tensor to a NumPy array
    similarity_matrix_np = sims.cpu().detach().numpy()[start:end, start:end]

    names = tacs[start:end]
    # Create the heatmap
    plt.figure(figsize=(10, 8))
    ax = sns.heatmap(similarity_matrix_np, xticklabels=names, yticklabels=names, annot=True, cmap='viridis')

    # Move x-axis labels to the top
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position('top')

    # Slant x-axis labels
    plt.xticks(rotation=45, ha='left')

    # Add labels and title
    plt.xlabel('Vectors')
    plt.ylabel('Vectors')
    plt.title('Pairwise Cosine Similarity Heatmap')

    # Display the heatmap
    plt.show()
   

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def plot_overlay(start, end, sims1, sims2):
    # Convert the tensors to NumPy arrays
    similarity_matrix_np1 = sims1.cpu().detach().numpy()[start:end, start:end]
    similarity_matrix_np2 = sims2.cpu().detach().numpy()[start:end, start:end]

    names = tacs[start:end]
    
    # Create the heatmap
    plt.figure(figsize=(10, 8))
    ax = sns.heatmap(similarity_matrix_np1, xticklabels=names, yticklabels=names, annot=False, cmap='viridis', cbar=False)
    sns.heatmap(similarity_matrix_np2, xticklabels=names, yticklabels=names, annot=False, cmap='viridis', cbar=True, ax=ax)
    
    # Move x-axis labels to the top
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position('top')

    # Slant x-axis labels
    plt.xticks(rotation=45, ha='left')

    # Add diagonal splits and annotations
    for i in range(len(names)):
        for j in range(len(names)):
            value1 = similarity_matrix_np1[i, j]
            value2 = similarity_matrix_np2[i, j]
            
            # Add diagonal line
            # ax.add_patch(patches.Polygon([(j, i), (j+1, i), (j, i+1)], closed=True, fill=True, color='blue', alpha=0.3))
            # ax.add_patch(patches.Polygon([(j+1, i), (j+1, i+1), (j, i+1)], closed=True, fill=True, color='red', alpha=0.3))

            ax.add_patch(patches.Polygon([(j, i), (j+1, i), (j, i+1)], closed=True, fill=True, facecolor=plt.cm.viridis(value1), alpha=0.3))
            ax.add_patch(patches.Polygon([(j+1, i), (j+1, i+1), (j, i+1)], closed=True, fill=True,facecolor=plt.cm.viridis(value2), alpha=0.3))
            # Add annotations
            ax.text(j + 0.25, i + 0.25, f'{value1:.2f}', ha='center', va='center', fontsize=8, color='white')
            ax.text(j + 0.75, i + 0.75, f'{value2:.2f}', ha='center', va='center', fontsize=8, color='white')
    
    # Add labels and title
    plt.xlabel('Tactic')
    plt.ylabel('Tactic')
    plt.title('Overlay Pairwise Cosine Similarity Heatmap')

    # Display the heatmap
    plt.show()

# Example usage
# plot_overlay(start, end, sims1, sims2)


In [ ]:
# ckpt_path = '../runs/diversity/with_tac/2024_07_05/16_34/checkpoints/tac_enc'

# combined_encoder_path = '../runs/diversity/large-single-vec-2/2024_07_15/10_57/checkpoints/combined_enc.ckpt'
# separate_encoder_path = '../runs/separate_tac_encoder/checkpoints/separate_encoder.ckpt'

# combined_encoder_path = '../runs/diversity/large-single-vec-2/2024_07_15/10_57/checkpoints/combined_last.ckpt'

# separate_encoder_path = '../runs/separate_tac_encoder/checkpoints/separate_last.ckpt'

autoenc_path = '../runs/diversity/autoencoder/2024_07_18/12_13/checkpoints/autoencoder.ckpt'

from models.end_to_end.tactic_models.error_pred.model import ErrorPredModel as ErrorModel

combined_error_path = '../runs/error_pred/combined_transition_model/combined_minif2f_valid.ckpt'


# ckpt = torch.load(combined_encoder_path)
# state_dict = {k[12:]: v for k, v in ckpt.items() if k.startswith('tac_encoder')}
# 
# tokenizer = AutoTokenizer.from_pretrained('sean-lamont/leandojo-lean3-reprover-novel-premises')
# 
# combined_encoder = T5EncoderModel.from_pretrained('sean-lamont/leandojo-lean3-reprover-novel-premises',
#                                                   state_dict=state_dict).cuda()


ckpt = torch.load(combined_error_path)
state_dict = {k[12:]: v for k, v in ckpt.items() if k.startswith('tac_encoder')}

error_encoder = T5EncoderModel.from_pretrained('sean-lamont/leandojo-lean3-reprover-novel-premises',
                                                  state_dict=state_dict).cuda()


# ckpt = torch.load(separate_encoder_path)
# 
# state_dict = {k[12:]: v for k, v in ckpt.items() if k.startswith('tac_encoder')}
# 
# separate_encoder = T5EncoderModel.from_pretrained('sean-lamont/leandojo-lean3-reprover-novel-premises',
#                                                   state_dict=state_dict).cuda()
# 
# 
# ckpt = torch.load(autoenc_path)
# 
# state_dict = {k[12:]: v for k, v in ckpt.items() if k.startswith('tac_encoder')}
# 
# autoencoder = T5EncoderModel.from_pretrained('sean-lamont/leandojo-lean3-reprover-novel-premises',
#                                                   state_dict=state_dict).cuda()



In [ ]:

score_network = torch.nn.Sequential(
    torch.nn.Linear(error_encoder.config.d_model, error_encoder.config.d_model // 2),
    torch.nn.LayerNorm(error_encoder.config.d_model // 2),
    torch.nn.ReLU(),
    torch.nn.Linear(error_encoder.config.d_model // 2, 2),
)

ckpt = torch.load(combined_error_path)
state_dict = {k[14:]: v for k, v in ckpt.items() if k.startswith('score_network')}

score_network.load_state_dict(state_dict)


In [ ]:
score_network.to('cuda')

In [ ]:
    
# good example with node 0
test_trace = '../runs/minif2f/minif2f_bestfs/2024_07_31/16_28_47/traces/0/aime_1984_p15'
# test_trace = "../runs/bestfs-novel-2/cau_seq.lim_inv"

trace = pickle.load(open(test_trace, 'rb'))
nodes = list(trace.nodes.values())

In [ ]:
trace

In [ ]:
node = nodes[0]
tacs = [e.tactic for e in node.out_edges]
theorem = trace.theorem.full_name

with torch.no_grad():
    combined_encs = get_vecs(combined_encoder, tacs, node, theorem)
    error_encs = get_vecs(error_encoder, tacs, node, theorem)

# with torch.no_grad():
#     tokenized_tacs = [tokenizer( e,
#                                 padding="longest",
#                                 max_length=2300,
#                                 truncation=True,
#                                 return_tensors="pt") for e in tacs]
# 
# 
#     lens = [tokenized_tactic.attention_mask.sum(dim=1) for tokenized_tactic in tokenized_tacs]
# 
# 
#     vecs = [_encode(separate_encoder, tac.input_ids.cuda(), tac.attention_mask.cuda())[0] for tac in tokenized_tacs]
#     separate_encs = torch.stack(vecs)
# 
#     autoencs = [_encode(autoencoder, tac.input_ids.cuda(), tac.attention_mask.cuda())[0] for tac in tokenized_tacs]
#     autoencs = torch.stack(autoencs)

# separate_cosine_sim = pairwise_cosine_similarity(separate_encs)


combined_cosine_sim = pairwise_cosine_similarity(combined_encs)
error_cosine_sim = pairwise_cosine_similarity(error_encs)


# autoencsim = pairwise_cosine_similarity(autoencs)


In [ ]:
scores = score_network(error_encs)

error_preds = torch.sigmoid(scores[:, 0])
time_scores = scores[:, 1]

# normalise time scores

time_scores = F.normalize(time_scores, dim=-1, p=2)
time_scores = 1 - time_scores



In [ ]:
time_scores

In [ ]:
from experiments.end_to_end.proof_node import ErrorNode

start = 0
end = 16

names = tacs[start:end]
gt = node.out_edges[start:end]
for x in [(i + start, names[i], round(error_preds[i].item(),2), 0 if type(gt[i].dst[0]) == ErrorNode else 1, time_scores[i].item(), gt[i].time, ) for i in range(len(names))]:
    print (x)


# print (node.data['augmented_state'])
plot_overlay(start, end, error_cosine_sim, combined_cosine_sim)
# plot_overlay(start, end, error_cosine_sim, time_scores.unsqueeze(1))

# plot_overlay(start, end, autoencsim, combined_cosine_sim)

In [ ]:
gt[0].dst, gt[1].dst

In [ ]:
## Combined error pred model vs just transitions:
## Chooses apparently dissimilar tactics as similar, most likely due to error message
## Happens mostly(?) when many errors for that node?
## 
